In [2]:
# Author: Ryan Benac CEMVR EC-D
# Last Update: 1/20/2026
# This script uses python playwright to upload data to EMS REST API from a spreadsheet
##################################################################################################################################
print(f"Importing modules...")
import pandas as pd # used to manage datatable
from playwright.sync_api import sync_playwright, TimeoutError # used to interact with EMS
import requests, json
from urllib.parse import quote
import datetime
import urllib3

# Disable the InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# variables
#baseEMSURL = "https://ems-test.cwbi.us"
# The 'r' before the string treats backslashes as literal characters
#file_path = r"C:\Workspace\LOCAL SANDBOX\EMS Rest API\Bulk Insert Tasks to EMS\TEST_DATASETS.xlsx" 
#sheet_name = "compiled"


# variables
baseEMSURL = "https://ems.sec.usace.army.mil/"
# The 'r' before the string treats backslashes as literal characters
file_path = r"C:\Workspace\LOCAL SANDBOX\EMS Rest API\Bulk Insert Tasks to EMS\Copy of CR-FRM EMS Products w P6 Tasks.xlsx"
sheet_name = "compiled"

print('Modules imported...')

Importing modules...
Modules imported...


In [9]:
##################################################################################################################################
# STEP 1: import spreadsheet as datatable
print(f"Importing spreadsheet at {file_path}...")
try:
    # Read the specified sheet from the Excel file into a pandas DataFrame
    # The 'engine='openpyxl'' is required for .xlsx files
    df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # Print a success message
    print(f"Successfully imported data from sheet '{sheet_name}'.")

except FileNotFoundError:
    print(f"Error: The file was not found at the path: {file_path}")
except Exception as e:
    # Catch other potential errors, such as the sheet not existing
    print(f"An error occurred: {e}")

# format dates as yyyy-mm-dd
# Convert date columns to datetime objects, coercing errors to NaT (Not a Time)
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

# Format the datetime objects into the desired 'yyyy-mm-dd' string format
df['start_date'] = df['start_date'].dt.strftime('%Y-%m-%d')
df['end_date'] = df['end_date'].dt.strftime('%Y-%m-%d')

# convert forward slashes to backward slashes
df['task_name'] = df['task_name'].str.replace('/', '\\', regex=False)

# URL-encode task_name values up-front and store in a column (encode slashes too)
df['task_name_encoded'] = df['task_name'].apply(lambda x: quote(str(x), safe='') if pd.notna(x) else '')

# Display the first 5 rows of the imported data to verify
print("First 5 rows of the DataTable:")
print(df.head())

Importing spreadsheet at C:\Workspace\LOCAL SANDBOX\EMS Rest API\Bulk Insert Tasks to EMS\Copy of CR-FRM EMS Products w P6 Tasks.xlsx...
Successfully imported data from sheet 'compiled'.
First 5 rows of the DataTable:
   product_id                                      product_title  master_task  \
0      132985  FCST CEMVR: CEDAR RAPIDS FRM - REACH 1 - MCLOU...            6   
1      132985  FCST CEMVR: CEDAR RAPIDS FRM - REACH 1 - MCLOU...            6   
2      132985  FCST CEMVR: CEDAR RAPIDS FRM - REACH 1 - MCLOU...            6   
3      132985  FCST CEMVR: CEDAR RAPIDS FRM - REACH 1 - MCLOU...            6   
4      132985  FCST CEMVR: CEDAR RAPIDS FRM - REACH 1 - MCLOU...            6   

   wbs_sub_id  wbs_id          nickname  \
0           1     6.1  R1 Middle McLoud   
1           2     6.2  R1 Middle McLoud   
2           3     6.3  R1 Middle McLoud   
3           4     6.4  R1 Middle McLoud   
4           5     6.5  R1 Middle McLoud   

                                    

In [13]:

sessionID = "8296799"
product_id = 427476
time_out = 30 # seconds
success_code = 200

unique_product_ids = df['product_id'].unique()
print(f"\nFound {len(unique_product_ids)} unique products to process.")

# filter to only current product id
sub_dataset = df[df['product_id'] == product_id]

# this section just gets the master task id to add new tasks to
print(f"\nProcessing Product ID {product_id} ({len(sub_dataset)} rows).")

scope_url = f"{baseEMSURL}/api/SCOPE/SSB_SCOPE/{sessionID}/{product_id}"
scope_data = None
try:
    print(f"   - Requesting scope: {scope_url}")
    resp = requests.get(scope_url, timeout=time_out, verify=False)
    if resp.status_code == 200:
        scope_data = resp.json()
        print(f"   - Success from {scope_url} (items={len(scope_data)})")
    else:
        print(f"   - Scope endpoint returned status {resp.status_code}: {scope_url}")
except Exception as e:
    print(f"   - Error requesting {scope_url}: {e}")
    # continue

if scope_data is None:
    print(f"   - Failed to retrieve scope for product {product_id}; skipping this product.")
    # continue

# there should only be 1 master task per product
master_wbs = str(sub_dataset['master_task'].unique()[0])

wbs1_entries = [s for s in scope_data if str(s.get('wbs')) == str(master_wbs)]
if not wbs1_entries:
    print("   - No scope entries with wbs == ':master_wbs' found. Skipping this product.")
    # continue

master_task_id = wbs1_entries[0].get('id')
taskName = wbs1_entries[0].get('lineItem')

print(f"   - Master task: id={master_task_id} and lineItem={taskName}")


Found 15 unique products to process.

Processing Product ID 427476 (74 rows).
   - Requesting scope: https://ems.sec.usace.army.mil//api/SCOPE/SSB_SCOPE/8296799/427476
   - Success from https://ems.sec.usace.army.mil//api/SCOPE/SSB_SCOPE/8296799/427476 (items=42)
   - Master task: id=7019742 and lineItem=CR-FRM Program P6 Task Structure


In [30]:
print(master_wbs)

7


In [14]:
# iterates over each row in the filtered dataset and creates task then adds start/end dates
insert_success_count = 0
insert_error = ""
start_cuccess_count = 0
start_error = ""
end_success_count = 0
end_error = ""
this_index = -1

for index, row in sub_dataset.iterrows():
    this_index = this_index + 1
    print(f"{datetime.datetime.now()} Processing row {this_index + 1}/{len(sub_dataset)}...")
    #print(row)
    # STEP 1 INSERT ROW
    try:
        task_name = row['task_name_encoded']
        #print(row['task_name_encoded'])
        wbs_sub_id = row['wbs_sub_id']
        #print(row['wbs_sub_id'])
        wbs_num = f"{master_wbs}.{wbs_sub_id}" # need this to filter response for new task id
        #print(wbs_num)

        print(f"     - Row {this_index + 1}: task='{task_name}', master_task_id={master_task_id}, sessionID={sessionID}, wbs_sub_id={wbs_sub_id}, wbs_num={wbs_num}")


        # Build insert URL and call it
        if master_task_id is None:
            print("     - No master_task_id available; skipping insert for this row.")
            insert_error = f"{insert_error}, {this_index+1}"
        elif not task_name:
            print("     - task_name is empty; skipping insert for this row.")
            insert_error = f"{insert_error}, {this_index+1}"
        else:
            insert_url = f"{baseEMSURL}/api/ssb_wbs/insert/{sessionID}/{product_id}/{master_task_id}/{task_name}/{wbs_sub_id}"
            try:
                resp = requests.get(insert_url, timeout=time_out, verify=False)
                scope_data = resp.json()
                print(f"     - Insert request -> status {resp.status_code}")
                # Optionally print response body for debugging
                # print(resp.text)
            except Exception as e:
                print(f"     - Insert request failed: {e}")
        # Build URLs with f-strings (avoid concat with &)
        # Example insert URL (uncomment and adapt when ready):
        # insert_url = f"{baseEMSURL}/api/ssb_wbs/insert/{sessionID}/{product_id}/{master_task_id}/{task_name}/1"
    except Exception as e:
        print(f"     - Error processing row {this_index + 1}: {e}")
        insert_error = f"{insert_error}, {this_index+1}"
    
    if resp.status_code == success_code:
        insert_success_count = insert_success_count + 1
    else: 
        insert_error = f"{insert_error}, {this_index+1}"

    
    
    # get task id to add start and end dates
    created_task_id = [s for s in scope_data if str(s.get('wbs')) == wbs_num]
    if not created_task_id:
        print("   - No scope entries with wbs == {wbs_num} found. Skipping this product.")
        insert_error = f"{insert_error}, {this_index+1}"
        continue

    task_id = created_task_id[0].get('id')


    # STEP 2 INSERT START AND END DATES
    try:
        start_date = row['start_date']
        end_date = row['end_date']
    
        # Build insert URL and call it
        if start_date is None or end_date is None:
            print("     - No start or end date available; skipping insert for this row.")
            start_error = f"{start_error}, {this_index+1}"
            end_error = f"{end_error}, {this_index+1}"
        elif not start_date or not end_date:
            print("     - Start or end date is empty; skipping insert for this row.")
            start_error = f"{start_error}, {this_index+1}"
            end_error = f"{end_error}, {this_index+1}"
        else:
            insert_start_url = f"{baseEMSURL}/api/ssb_task_overrides/updateSTARTDATE/{sessionID}/{product_id}/{task_id}/{start_date}"
            try:
                resp = requests.get(insert_start_url, timeout=time_out, verify=False)
                print(f"     - Insert start date request -> status {resp.status_code}")
                # Optionally print response body for debugging
                # print(resp.text)
            except Exception as e:
                print(f"     - Insert start date request failed: {e}")
            
            if resp.status_code == success_code:
                start_cuccess_count = start_cuccess_count + 1
            else: 
                start_error = f"{start_error}, {this_index+1}"

            insert_end_url = f"{baseEMSURL}/api/ssb_task_overrides/updateENDDATE/{sessionID}/{product_id}/{task_id}/{end_date}"
            try:
                resp = requests.get(insert_end_url, timeout=time_out, verify=False)
                print(f"     - Insert end date request -> status {resp.status_code}")
                # Optionally print response body for debugging
                # print(resp.text)
            except Exception as e:
                print(f"     - Insert end date request failed: {e}")

            if resp.status_code == success_code:
                end_success_count = end_success_count + 1
            else: 
                end_error = f"{end_error}, {this_index+1}"
    
    except Exception as e:
        print(f"     - Error processing row {this_index + 1}: {e}")
        start_error = f"{start_error}, {this_index+1}"
        end_error = f"{end_error}, {this_index+1}"



# FINAL SUMMARY
print(f"\n\n===== FINAL SUMMARY =====")
print(f"Total rows processed for product {product_id}: {len(sub_dataset)}")
print(f" - Inserted tasks successfully: {insert_success_count}")
print(f" - Insert task errors on rows: {insert_error if insert_error else 'None'}")
print(f" - Inserted start dates successfully: {start_cuccess_count}")
print(f" - Insert start date errors on rows: {start_error if start_error else 'None'}")
print(f" - Inserted end dates successfully: {end_success_count}")
print(f" - Insert end date errors on rows: {end_error if end_error else 'None'}")



2026-01-26 07:46:01.613763 Processing row 1/74...
     - Row 1: task='LTC2R-RE2600__Real%20Estate%20Certification', master_task_id=7019742, sessionID=8296799, wbs_sub_id=1, wbs_num=5.1
     - Insert request -> status 200
     - Insert start date request -> status 200
     - Insert end date request -> status 200
2026-01-26 07:46:08.218982 Processing row 2/74...
     - Row 2: task='LTC2R-RE2200__Condemnation', master_task_id=7019742, sessionID=8296799, wbs_sub_id=2, wbs_num=5.2
     - Insert request -> status 200
     - Insert start date request -> status 200
     - Insert end date request -> status 200
2026-01-26 07:46:15.934587 Processing row 3/74...
     - Row 3: task='LTC2R-RE1950__Negotiations', master_task_id=7019742, sessionID=8296799, wbs_sub_id=3, wbs_num=5.3
     - Insert request -> status 200
     - Insert start date request -> status 200
     - Insert end date request -> status 200
2026-01-26 07:46:25.251798 Processing row 4/74...
     - Row 4: task='LTC2R-RE1550__Appraisal',